<a href="https://colab.research.google.com/github/rxphaelbihag/Linear-Programming/blob/main/Store_Location_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Store Location Optimization

DS115: End-of-Sem Project

## Description
In this project, you are an entrepreneur with a budget enough to open 2 convenience stores in downtown Davao. The idea is to find the best locations of only two convenience stores that would allow more access to your target clients. You are provided 7 locations to choose from and you are given the locations of your 5 target clients.

## Specifications
1. The coordinates of the 7 possible store locations are stored in a .CSV file.
2. The coordinates of the 5 clients are stored in a .CSV file.
3. Randomly pick two pairs of possible store locations.
4. Using p-center as the math model for this problem, determine which of the
two sets should you choose as locations for your stores

## Calculations
### Libraries and CSV files
We use `pandas` as our main tool for data manipulation. The `cdist` class from `scipy` allows us to calculate eucledian distances. We use `random` to generate random numbers. We use `folium` to generate the map of the final answer.

In [212]:
import pandas as pd
from scipy.spatial.distance import cdist
import random
import folium

### Import the CSV Files

In [213]:
store_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/bihag_storelocs.csv"
client_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/client_locs.csv"

stores = pd.read_csv(store_locations_csv, index_col='store')
clients = pd.read_csv(client_locations_csv, index_col='client')

In [214]:
stores

,latitude,longitude
store,,
1,7.090138,125.606766
2,7.088724,125.630833
3,7.081656,125.612843
4,7.081869,125.621222
5,7.079220,125.625024
6,7.072015,125.602871
7,7.065656,125.610641


In [215]:
clients

,latitude,longitude
client,,
1,7.086733,125.621816
2,7.082687,125.615703
3,7.077230,125.604769
4,7.085151,125.607086
5,7.073291,125.611879


### Construct Distance Matrix

We use a distance matrix to calculate the distance of each client to each store. This is so that it will be easier for calculations later.

In [216]:
# Extract coordinate columns as NumPy arrays
client_coords = clients[['latitude', 'longitude']].values
store_coords = stores[['latitude', 'longitude']].values

# Compute the pairwise distance matrix
distance_array = cdist(client_coords, store_coords, metric='euclidean')

# Convert the array back into a labeled Pandas DataFrame
distance_matrix = pd.DataFrame(
    distance_array,
    index=clients.index,  # Rows represent clients
    columns=stores.index  # Columns represent stores
)

distance_matrix

store,1,2,3,4,5,6,7
client,,,,,,,
1,0.015430,0.009234,0.010310,0.004900,0.008170,0.023990,0.023856
2,0.011636,0.016290,0.003040,0.005579,0.009945,0.016689,0.017767
3,0.013062,0.028486,0.009207,0.017094,0.020352,0.005550,0.012978
4,0.004997,0.024014,0.006735,0.014511,0.018893,0.013796,0.019816
5,0.017606,0.024442,0.008421,0.012684,0.014420,0.009097,0.007734


### Random Two Sets
We randomly pick two sets or two pairs of store locations. The store locatiosn are indexed from 1 to 7 (Store 1 to Store 7). So, we use `random`'s `sample()` function to sample four random numbers and group them to two. This is to ensure that the two sets do not have the same stores.

In [217]:
numbers = random.sample(range(1, 8), 4)

# The sets in index form
set1 = (numbers[0], numbers[1])
set2 = (numbers[2], numbers[3])

# The sets in coordinates form
set1_coords = [store_coords[set1[0]-1], store_coords[set1[1]-1]]
set2_coords = [store_coords[set2[0]-1], store_coords[set2[1]-1]]

print(f"Set 1: {set1}", set1_coords)
print(f"Set 2: {set2}", set2_coords)

Set 1: (5, 1) [array([  7.07921956, 125.62502397]), array([  7.09013824, 125.60676599])]
Set 2: (2, 6) [array([  7.0887236 , 125.63083284]), array([  7.07201496, 125.60287135])]


### Evaluation
Next, we evaluate the distances of each of the clients to each of the stores in our two random sets. This will be very easy since we have already calculated all of the pairwise distances and stored in the `distance_matrix`. Now, we just have to evaluate each client's distance to the two stores in a set to see which store is closest to it and by how much.

In [218]:
# Evaluating first set

# Stores the distances. Format: "Client 1": [<Distance>, <Closest Store>]
# Distance is in degrees
# Closest Store is either 0 if closest to the first store in the set, 1 if
# otherwise
set1_dists = {}

for client in range(1, 5+1):
    store1 = set1[0]    # coords of store1
    store2 = set1[1]    # coords of store2

    distto_s1 = distance_matrix[store1][client] # dist of client to store 1
    distto_s2 = distance_matrix[store2][client] # dist of client to store 2

    # find the minimum among the two distances
    if distto_s1 > distto_s2:
        set1_dists[f"client{client}"] = (distto_s2, 1)
    else:
        set1_dists[f"client{client}"] = (distto_s1, 0)

In [219]:
# Evaluating second set

# Stores the distances. Format: "Client 1": [<Distance>, <Closest Store>]
# Distance is in degrees
# Closest Store is either 0 if closest to the first store in the set, 1 if
# otherwise
set2_dists = {}

for client in range(1, 5+1):
    store1 = set2[0]    # coords of store1
    store2 = set2[1]    # coords of store2

    distto_s1 = distance_matrix[store1][client] # dist of client to store 1
    distto_s2 = distance_matrix[store2][client] # dist of client to store 2

    # find the minimum among the two distances
    if distto_s1 > distto_s2:
        set2_dists[f"client{client}"] = (distto_s2, 1)
    else:
        set2_dists[f"client{client}"] = (distto_s1, 0)

In [220]:
# Evaluate which set is the best
if max(set1_dists.values())[0] > max(set2_dists.values())[0]:
    best = set2
    print("The best set is Set 2:", set2, "The max distance (deg) is", max(set2_dists.values())[0])
else:
    best = set1
    print("The best set is Set 1:", set1, "The max distance (deg) is", max(set1_dists.values())[0])

The best set is Set 1: (5, 1) The max distance (deg) is 0.014420401815248274


## Optimal Solution

Now that we have identified the best set. We plot them on a map to visualize it.

In [225]:
# Initialize map center (Somewhere in downtown Davao City)
mymap = folium.Map(location=[7.080506387226319, 125.6120560701777],
                   zoom_start=12)

# dictionary of client coordinates
clients = {}
for _ in range(len(client_coords)):
    clients[f"Client {_+1}"] = [client_coords[_][0], client_coords[_][1]]

# dictionary of store coordinates in the best set
best_set = {}
for _ in range(2):
    best_set[f"Store {best[_]}"] = [store_coords[best[_]-1][0],
                                    store_coords[best[_]-1][1]]

for client, coords in clients.items():
    folium.Marker(
        location=coords,
        popup=client,
        icon=folium.Icon(color="blue", icon="person", prefix='fa')
    ).add_to(mymap)

for store, coords in best_set.items():
    folium.Marker(
        location=coords,
        popup=store,
        icon=folium.Icon(color="red", icon="store", prefix='fa')
    ).add_to(mymap)

    folium.Circle(
        location=coords,
        radius=(max(set1_dists.values()))[0]*111320, # Radius in meters
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.2,
        popup="Coverage Area"
    ).add_to(mymap)

mymap